<a href="https://colab.research.google.com/github/AmmarNasirEngr/Traditional-Vector-RAG-vs-Page-Index-RAG/blob/main/Traditional_RAG_vs_PageIndex_RAG_By_Ammar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG Comparison: Traditional Vector RAG vs Page Index RAG

This notebook demonstrates a comparison between Traditional RAG and Page Index RAG for document retrieval and question answering. It explores their mechanisms, strengths, and weaknesses through practical examples.

The author of this notebook is Ammar Nasir.

**100% self-contained — just open this in Colab and run every cell top to bottom.**

No external files, no GitHub clones, no folder structure needed.
All pipeline code is written inline in the cells below.

---

## Pipeline overview

```
TRADITIONAL RAG               PAGE INDEX RAG
────────────────              ─────────────────────
Document                      Document
   ↓                             ↓
Split into fixed chunks       LLM reads pages
   ↓                             ↓
Embed each chunk              Build TOC tree
   ↓                             ↓
Store in vector DB            TOC stored in memory

User query                    User query
   ↓                             ↓
Embed query                   LLM traverses tree
   ↓                             ↓
Cosine similarity search      Fetch exact source pages
   ↓                             ↓
Top-K chunks → LLM            Source pages → LLM
   ↓                             ↓
Answer                        Answer
```

## Table of contents
1. Install & API key
2. Shared utilities
3. Sample document
4. Traditional RAG pipeline
5. Page Index RAG pipeline
6. Run & compare both pipelines
7. Experiments
8. Try your own PDF


## Cell 1 — Install dependencies

In [1]:
!pip install openai anthropic pypdf -q
print('✓ Done')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 763.1/763.1 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 19.5 MB/s eta 0:00:00
✓ Done


## Cell 2 — Set your API key

**Recommended**: Left sidebar → 🔑 Secrets → add `OPENAI_API_KEY`  
Then uncomment the `userdata` line below.

Or paste your key directly into the string (less secure).

In [2]:
import os

# Option A — Colab Secrets (recommended)
# from google.colab import userdata
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# Option B — paste directly
# os.environ['OPENAI_API_KEY'] = 'sk-...'

# ── Anthropic instead of OpenAI? ──────────────────────────────────────────
# Uncomment the line below AND set PROVIDER = 'anthropic' in Cell 3
# os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

os.environ['NVIDIA_API_KEY'] = "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"

key = os.getenv('NVIDIA_API_KEY', '')
print(f'✓ NVIDIA_API_KEY set: {key[:12]}...')

✓ NVIDIA_API_KEY set: nvapi-YK6g-k...


## Cell 3 — Global config

Change these values to switch providers, models, or chunk settings.

In [23]:
# ── Provider & model ───────────────────────────────────────────────────────
PROVIDER         = 'nvidia'
LLM_MODEL        = 'meta/llama-3.3-70b-instruct'   # ← change this line
EMBEDDING_MODEL  = 'nvidia/nv-embedqa-e5-v5'        # free NVIDIA embedding model

# ── Traditional RAG settings ───────────────────────────────────────────────
CHUNK_SIZE  = 800     # characters per chunk — try 300, 800, 1500
OVERLAP     = 100     # overlap between chunks — try 0, 50, 150
TOP_K       = 5       # how many chunks to retrieve per query

# ── Page Index settings ────────────────────────────────────────────────────
CHARS_PER_PAGE = 2000  # simulated page size
BATCH_SIZE     = 3     # pages sent to LLM per indexing call

print('✓ Config set')
print(f'  Provider : {PROVIDER}  |  Model : {LLM_MODEL}')
print(f'  Chunk size: {CHUNK_SIZE}  |  Overlap: {OVERLAP}  |  Top-K: {TOP_K}')

✓ Config set
  Provider : nvidia  |  Model : meta/llama-3.3-70b-instruct
  Chunk size: 800  |  Overlap: 100  |  Top-K: 5


## Cell 4 — Shared utilities

Helper functions used by both pipelines:
- `load_document()` — load .txt or .pdf
- `split_into_pages()` — simulate pages from plain text
- `estimate_tokens()` — rough token count
- `Timer` — measure latency
- `QueryResult` — structured result holder
- `LLMClient` / `EmbeddingClient` — unified API wrappers

In [24]:
# ════════════════════════════════════════════════════════════
# SHARED UTILITIES
# ════════════════════════════════════════════════════════════

import os, math, time, json, textwrap
from dataclasses import dataclass, field
from pathlib import Path


# ── Result container ────────────────────────────────────────

@dataclass
class QueryResult:
    pipeline: str
    query: str
    answer: str
    retrieved_context: str
    latency_seconds: float
    estimated_tokens_used: int
    metadata: dict = field(default_factory=dict)

    def summary(self):
        print('\n' + '='*62)
        print(f'  Pipeline : {self.pipeline.upper()}')
        print(f'  Query    : {self.query}')
        print(f'  Latency  : {self.latency_seconds:.2f}s')
        print(f'  Tokens   : ~{self.estimated_tokens_used:,}')
        print('  ' + '-'*58)
        print('  Answer:')
        for line in textwrap.wrap(self.answer, width=60):
            print(f'    {line}')
        print('='*62)


# ── Document loading ────────────────────────────────────────

def load_document(path: str) -> str:
    ext = Path(path).suffix.lower()
    if ext == '.pdf':
        from pypdf import PdfReader
        reader = PdfReader(path)
        return '\n\n'.join(p.extract_text() or '' for p in reader.pages)
    return open(path, encoding='utf-8').read()

def split_into_pages(text: str, chars_per_page: int = 2000) -> list:
    pages = []
    for i in range(0, len(text), chars_per_page):
        page = text[i:i+chars_per_page].strip()
        if page:
            pages.append(page)
    return pages


# ── Token & cost estimation ─────────────────────────────────

def estimate_tokens(text: str) -> int:
    return max(1, len(text) // 4)


# ── Timer ───────────────────────────────────────────────────

class Timer:
    def __enter__(self):
        self._start = time.perf_counter()
        return self
    def __exit__(self, *a):
        self.elapsed = time.perf_counter() - self._start


# ── LLM client ──────────────────────────────────────────────

class LLMClient:
    def __init__(self, provider=PROVIDER, model=LLM_MODEL, temperature=0.0, max_tokens=1024):
        self.provider = provider.lower()
        self.model = model
        self.max_tokens = max_tokens
        from openai import OpenAI
        self._c = OpenAI(
            base_url="https://integrate.api.nvidia.com/v1",
            api_key=os.getenv('NVIDIA_API_KEY'),
        )

    def chat(self, user_prompt: str, system: str = 'You are a helpful assistant.') -> str:
        response_text = ''
        completion = self._c.chat.completions.create(
            model=self.model,
            messages=[
                {'role': 'system', 'content': system},
                {'role': 'user',   'content': user_prompt},
            ],
            temperature=0.6,
            top_p=0.95,
            max_tokens=self.max_tokens,
            stream=True,
        )
        for chunk in completion:
            if not chunk.choices:
                continue
            delta = chunk.choices[0].delta
            if getattr(delta, 'reasoning_content', None):
                continue
            if delta.content:
                response_text += delta.content
        return response_text.strip()


# ── Embedding client ─────────────────────────────────────────

class EmbeddingClient:
    def __init__(self, model=EMBEDDING_MODEL):
        from openai import OpenAI
        self._c = OpenAI(
            base_url="https://integrate.api.nvidia.com/v1",
            api_key=os.getenv('NVIDIA_API_KEY'),
        )
        self.model = model

    def embed(self, text: str) -> list:
        r = self._c.embeddings.create(
            model=self.model,
            input=[text],
            encoding_format='float',
            extra_body={"input_type": "query", "truncate": "END"}
        )
        return r.data[0].embedding

    def embed_batch(self, texts: list) -> list:
        results, batch_size = [], 50
        for i in range(0, len(texts), batch_size):
            r = self._c.embeddings.create(
                model=self.model,
                input=texts[i:i+batch_size],
                encoding_format='float',
                extra_body={"input_type": "passage", "truncate": "END"}
            )
            results.extend(item.embedding for item in r.data)
        return results


# ── Pretty comparison printer ────────────────────────────────

def print_comparison(ra: QueryResult, rb: QueryResult):
    print('\n' + '═'*62)
    print('  SIDE-BY-SIDE COMPARISON')
    print('═'*62)
    rows = [
        ('Latency', f"{ra.latency_seconds:.2f}s",      f"{rb.latency_seconds:.2f}s"),
        ('Tokens',  f"~{ra.estimated_tokens_used:,}", f"~{rb.estimated_tokens_used:,}"),
    ]
    print(f'  {"Metric":<12} {"Traditional RAG":<22} {"Page Index RAG"}')
    print('  ' + '-'*55)
    for label, a, b in rows:
        print(f'  {label:<12} {a:<22} {b}')
    print(f'\n  Query: {ra.query}')
    print('\n  ── Traditional RAG answer ──')
    for line in textwrap.wrap(ra.answer, 60):
        print(f'  {line}')
    print('\n  ── Page Index RAG answer ──')
    for line in textwrap.wrap(rb.answer, 60):
        print(f'  {line}')
    print('='*62)


print('✓ Shared utilities loaded')

✓ Shared utilities loaded


## Cell 5 — Sample document

A synthetic AI textbook is embedded directly in this cell — no file upload needed.  
It covers: ML fundamentals, deep learning, NLP, computer vision, AI in industry, ethics.

You can replace `DOCUMENT_TEXT` with any string (or load your own PDF in Cell 14).

In [14]:
DOCUMENT_TEXT = """
ARTIFICIAL INTELLIGENCE IN MODERN APPLICATIONS — A Comprehensive Guide

CHAPTER 1: INTRODUCTION TO AI

Artificial Intelligence (AI) refers to the simulation of human intelligence processes by computer systems.
These processes include learning, reasoning, and self-correction. The term was first coined by John McCarthy
in 1956 at the Dartmouth Conference, widely considered the birthplace of AI as a field.

AI systems work by ingesting large amounts of labeled training data, analyzing the data for correlations and
patterns, and using these patterns to make predictions about future states. The primary goals of AI include
Natural Language Processing, Expert Systems, Computer Vision, and Robotics.

Modern AI is divided into Narrow AI (ANI), which performs specific tasks like facial recognition, and
General AI (AGI), the concept of machines that can perform any intellectual task a human can.

CHAPTER 2: MACHINE LEARNING FUNDAMENTALS

Machine Learning (ML) is a subset of AI that provides systems the ability to learn and improve from
experience without being explicitly programmed. ML focuses on developing programs that can access data
and learn from it autonomously.

SECTION 2.1 — SUPERVISED LEARNING

In supervised learning, the algorithm is trained on labeled data. The model learns a mapping from inputs
to outputs based on example input-output pairs. Key algorithms include Linear Regression, Logistic
Regression, Decision Trees, Random Forest, Support Vector Machines (SVM), and Neural Networks.

The training process adjusts model parameters to minimize a loss function. Common loss functions include
Mean Squared Error (MSE) for regression and Cross-Entropy for classification tasks.

SECTION 2.2 — UNSUPERVISED LEARNING

Unsupervised learning analyzes and clusters unlabeled datasets. These algorithms discover hidden patterns
without human intervention. Key techniques include K-Means Clustering, DBSCAN, Principal Component
Analysis (PCA), Autoencoders, and Generative Adversarial Networks (GANs).

SECTION 2.3 — REINFORCEMENT LEARNING

Reinforcement Learning (RL) trains an agent to make decisions by interacting with an environment.
The agent receives rewards for correct actions and penalties for incorrect ones, aiming to maximize
cumulative reward over time.

Key RL concepts: Agent (the learner), Environment (what the agent interacts with), State (current
situation), Action (what the agent can do), Reward (feedback signal), Policy (the agent's strategy),
and Value Function (expected cumulative reward from a state).

Notable RL algorithms: Q-Learning, Deep Q-Network (DQN), Proximal Policy Optimization (PPO), and
Actor-Critic methods. RL has been applied to games (AlphaGo), robotics, and recommendation systems.

CHAPTER 3: DEEP LEARNING AND NEURAL NETWORKS

Deep learning uses neural networks with many layers to learn from large amounts of data. It has driven
remarkable improvements in computer vision, natural language processing, and many other fields.

SECTION 3.1 — FEEDFORWARD NETWORKS

A feedforward neural network is the simplest type of artificial neural network. Information moves in
only one direction — forward — from input nodes through hidden nodes to output nodes. Components include
Input Layer, Hidden Layers (with activation functions like ReLU, Sigmoid, Tanh), Output Layer, and
Backpropagation for training by adjusting weights using gradients.

SECTION 3.2 — CONVOLUTIONAL NEURAL NETWORKS (CNNs)

CNNs are designed for processing grid data like images. They use convolutional layers that apply filters
to detect local features such as edges, textures, and shapes. Key layers: Convolutional Layer, Pooling
Layer (max/average pooling), Fully Connected Layer, Batch Normalization.

Famous CNN architectures: AlexNet, VGG, ResNet, Inception, EfficientNet. CNNs are the standard
approach for medical image analysis, satellite imaging, and quality inspection in manufacturing.

SECTION 3.3 — RECURRENT NEURAL NETWORKS (RNNs)

RNNs are designed for sequential data. Unlike feedforward networks, RNNs have connections that form
directed cycles, allowing information to persist across time steps. Suitable for language modeling,
time series prediction, and speech recognition.

RNN variants: LSTM (Long Short-Term Memory) — solves vanishing gradient; GRU (Gated Recurrent Unit)
— simplified LSTM; Bidirectional RNNs — process sequences in both directions simultaneously.

CHAPTER 4: NATURAL LANGUAGE PROCESSING

NLP is the branch of AI concerned with giving machines the ability to understand and generate human
language. It draws from computer science and computational linguistics.

SECTION 4.1 — TOKENIZATION AND EMBEDDINGS

Tokenization breaks text into individual tokens (words, subwords, or characters). Word embeddings
represent words as dense vectors in a continuous space where similar words are nearby.

Key embedding methods: Word2Vec (learns from co-occurrence), GloVe (Global Vectors), FastText
(handles out-of-vocabulary using subwords), BERT and GPT (contextual embeddings that change based
on surrounding words).

SECTION 4.2 — TRANSFORMERS AND ATTENTION

The Transformer architecture, introduced in 'Attention is All You Need' (Vaswani et al., 2017),
revolutionized NLP. It relies on attention mechanisms to draw global dependencies between input
and output, removing the need for recurrence.

The attention mechanism allows the model to focus on relevant parts of the input when producing
each output token. Self-attention lets the model relate different positions within a single sequence.

Key Transformer components: Multi-Head Attention (attention applied multiple times in parallel),
Positional Encoding (injects position information), Feed-Forward Networks, Layer Normalization,
and Residual Connections for training deeper models.

SECTION 4.3 — LARGE LANGUAGE MODELS

Large Language Models (LLMs) are trained on vast text corpora and can generate human-like text,
answer questions, summarize documents, translate languages, and perform many other language tasks.

Key LLMs: GPT series (OpenAI), BERT (Google), Claude (Anthropic — Constitutional AI), LLaMA (Meta
— open source), Gemini (Google — multimodal). LLMs are trained via pre-training on large corpora
followed by fine-tuning using supervised learning or RLHF (Reinforcement Learning from Human Feedback).

CHAPTER 5: COMPUTER VISION

Computer vision deals with how computers gain understanding from digital images and videos, automating
tasks that the human visual system performs naturally.

SECTION 5.1 — OBJECT DETECTION

Object detection identifies and localizes objects within images using bounding box coordinates.
Unlike classification, it handles multiple objects simultaneously.

Popular detection algorithms: YOLO (You Only Look Once — real-time, processes full image at once),
SSD (Single Shot Detector — multi-scale), Faster R-CNN (region proposal network), DETR (Detection
Transformer — attention-based).

SECTION 5.2 — IMAGE SEGMENTATION

Image segmentation divides images into meaningful regions at pixel level. Types include Semantic
Segmentation (class label per pixel), Instance Segmentation (distinguishes individual objects),
and Panoptic Segmentation (combines both). Key models: U-Net, DeepLab, Mask R-CNN, SAM (Segment
Anything Model from Meta).

CHAPTER 6: AI IN INDUSTRY

SECTION 6.1 — HEALTHCARE

AI is transforming healthcare through improved diagnostics, drug discovery, and personalized medicine.
Medical Imaging: CNNs detect cancers, fractures, and abnormalities in X-rays, MRIs, and CT scans
with accuracy matching or exceeding radiologists. Drug Discovery: ML models predict molecular
properties and identify drug candidates faster than traditional methods. The FDA has approved over
500 AI-enabled medical devices as of 2024, spanning radiology, cardiology, and pathology.

SECTION 6.2 — FINANCE AND TRADING

Financial institutions use AI for fraud detection, algorithmic trading, credit scoring, and customer
service. Algorithmic Trading executes trades based on market patterns at high speed. Fraud Detection
performs real-time analysis of transactions. Robo-Advisors provide automated investment management.

SECTION 6.3 — AUTONOMOUS VEHICLES

Self-driving cars integrate multiple AI technologies. Perception uses cameras, LiDAR, and radar
combined with computer vision. Prediction anticipates behavior of other road users. Levels of
autonomy range from Level 0 (no automation) to Level 5 (full automation). Companies leading
development include Waymo, Tesla, and Cruise.

CHAPTER 7: ETHICS AND FUTURE OF AI

BIAS AND FAIRNESS: AI systems can perpetuate and amplify existing biases in training data. Fairness-
aware ML, bias auditing, and diverse training data are being developed to address this.

PRIVACY: AI requires large amounts of personal data. Privacy-preserving techniques include federated
learning, differential privacy, and secure multi-party computation.

TRANSPARENCY: Many AI models are black boxes. Explainable AI (XAI) techniques include LIME, SHAP,
and attention visualization to make decisions more interpretable.

JOB DISPLACEMENT: The World Economic Forum estimates AI will displace 85 million jobs but create
97 million new ones by 2025, with net positive employment outcomes expected.

FUTURE TRENDS: Multimodal AI (text + images + audio), AI Agents (autonomous action-taking systems),
Neuromorphic Computing (brain-inspired hardware), Quantum AI, and the long-term pursuit of
Artificial General Intelligence (AGI).
"""

pages = split_into_pages(DOCUMENT_TEXT, chars_per_page=CHARS_PER_PAGE)

print(f'✓ Document loaded')
print(f'  Characters : {len(DOCUMENT_TEXT):,}')
print(f'  Words      : {len(DOCUMENT_TEXT.split()):,}')
print(f'  Est. tokens: ~{estimate_tokens(DOCUMENT_TEXT):,}')
print(f'  Pages      : {len(pages)} (at {CHARS_PER_PAGE} chars each)')

✓ Document loaded
  Characters : 9,506
  Words      : 1,302
  Est. tokens: ~2,376
  Pages      : 5 (at 2000 chars each)


## Cell 6 — Traditional RAG: Chunker + Vector Store

This cell defines:
- `TextChunker` — splits document into fixed-size overlapping chunks
- `InMemoryVectorStore` — stores embeddings and does cosine similarity search

Read the comments — they explain exactly what each part does and why.

In [25]:
# ════════════════════════════════════════════════════════════
# TRADITIONAL RAG — CHUNKER + VECTOR STORE
# ════════════════════════════════════════════════════════════

@dataclass
class Chunk:
    """A single text chunk with its embedding."""
    chunk_id: int
    text: str
    start_char: int
    end_char: int
    embedding: list = field(default_factory=list)


class TextChunker:
    """
    Splits a document into fixed-size overlapping text chunks.

    chunk_size: target characters per chunk
    overlap   : characters shared between neighbouring chunks
                Overlap reduces hard context cuts at boundaries.
    """
    def __init__(self, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
        self.chunk_size = chunk_size
        self.overlap = overlap

    def chunk(self, text: str) -> list:
        chunks, chunk_id = [], 0
        step = self.chunk_size - self.overlap
        for start in range(0, len(text), step):
            end = min(start + self.chunk_size, len(text))
            chunk_text = text[start:end].strip()
            if len(chunk_text) < 20:
                break
            chunks.append(Chunk(chunk_id, chunk_text, start, end))
            chunk_id += 1
            if end == len(text):
                break
        return chunks

    def chunk_by_paragraph(self, text: str) -> list:
        """Alternative: split on double newlines."""
        paras = [p.strip() for p in text.split('\n\n') if p.strip()]
        chunks, pos = [], 0
        for i, para in enumerate(paras):
            start = text.find(para, pos)
            chunks.append(Chunk(i, para, start, start + len(para)))
            pos = start + len(para)
        return chunks


class InMemoryVectorStore:
    """
    Minimal vector store backed by a Python list.
    Uses cosine similarity for retrieval.

    Why not Pinecone/ChromaDB?
    → No extra API key, runs offline, and the maths is identical.
    """
    def __init__(self):
        self._chunks = []

    def add(self, chunks: list):
        self._chunks.extend(chunks)
        print(f'  ✓ Stored {len(chunks)} chunks (total: {len(self._chunks)})')

    def similarity_search(self, query_embedding: list, top_k: int = TOP_K) -> list:
        """
        Cosine similarity: cos(θ) = (A·B) / (|A||B|)
        Returns the top_k chunks most similar to the query.
        """
        scored = [
            (self._cosine(query_embedding, c.embedding), c)
            for c in self._chunks
        ]
        scored.sort(key=lambda x: x[0], reverse=True)
        print(f'\n  Top-{top_k} similarity scores:')
        for score, chunk in scored[:top_k]:
            preview = chunk.text[:55].replace('\n', ' ')
            print(f'    [{score:.4f}] Chunk {chunk.chunk_id}: {preview!r}…')
        return [c for _, c in scored[:top_k]]

    @staticmethod
    def _cosine(a, b):
        dot = sum(x*y for x, y in zip(a, b))
        na  = math.sqrt(sum(x**2 for x in a))
        nb  = math.sqrt(sum(x**2 for x in b))
        return dot / (na * nb) if na and nb else 0.0

    def __len__(self):
        return len(self._chunks)


print('✓ Traditional RAG components loaded (TextChunker, InMemoryVectorStore)')

✓ Traditional RAG components loaded (TextChunker, InMemoryVectorStore)


## Cell 7 — Traditional RAG: Full pipeline class

In [26]:
# ════════════════════════════════════════════════════════════
# TRADITIONAL RAG — FULL PIPELINE
# ════════════════════════════════════════════════════════════

class TraditionalRAGPipeline:
    """
    Full Traditional Vector RAG pipeline.

    Usage:
        pipeline = TraditionalRAGPipeline()
        pipeline.index(DOCUMENT_TEXT)
        result = pipeline.query('What is reinforcement learning?')
        result.summary()
    """

    def __init__(self, chunk_size=CHUNK_SIZE, overlap=OVERLAP, top_k=TOP_K):
        self.top_k       = top_k
        self.chunker     = TextChunker(chunk_size, overlap)
        self.vector_store = InMemoryVectorStore()
        self.llm         = LLMClient()
        self.embedder    = EmbeddingClient()
        self._indexed    = False

    # ── INDEXING PHASE ─────────────────────────────────────

    def index(self, text: str, strategy: str = 'fixed'):
        """
        INDEXING PHASE
        Step 1: Chunk the document
        Step 2: Embed each chunk via OpenAI embeddings API
        Step 3: Store vectors in memory
        """
        print('\n' + '='*52)
        print('  INDEXING PHASE — Traditional RAG')
        print('='*52)

        print('\n[1/3] Chunking document...')
        chunks = (self.chunker.chunk_by_paragraph(text)
                  if strategy == 'paragraph' else self.chunker.chunk(text))
        avg = sum(len(c.text) for c in chunks) // len(chunks)
        print(f'  ✓ {len(chunks)} chunks created | avg size: {avg} chars')
        print(f'  Sample (chunk 0): {chunks[0].text[:180]!r}…')

        print(f'\n[2/3] Embedding {len(chunks)} chunks...')
        embeddings = self.embedder.embed_batch([c.text for c in chunks])
        for chunk, emb in zip(chunks, embeddings):
            chunk.embedding = emb
        print(f'  ✓ Embedding dimensions: {len(embeddings[0])}')

        print('\n[3/3] Storing in vector store...')
        self.vector_store.add(chunks)

        self._indexed = True
        print('\n  ✓ Indexing complete!')

    # ── QUERY PHASE ────────────────────────────────────────

    def query(self, question: str) -> QueryResult:
        """
        QUERY PHASE
        Step 1: Embed the user question
        Step 2: Cosine similarity search → top-K chunks
        Step 3: Pass chunks + question to LLM
        """
        if not self._indexed:
            raise RuntimeError('Call .index(text) first.')

        print('\n' + '='*52)
        print('  QUERY PHASE — Traditional RAG')
        print('='*52)
        print(f'  Question: {question}')

        with Timer() as t:
            print('\n[1/3] Embedding query...')
            q_emb = self.embedder.embed(question)

            print(f'\n[2/3] Similarity search (top-{self.top_k})...')
            retrieved = self.vector_store.similarity_search(q_emb, self.top_k)

            context = '\n\n---\n\n'.join(c.text for c in retrieved)
            ctx_tokens = estimate_tokens(context)

            print(f'\n[3/3] Generating answer (~{ctx_tokens} context tokens)...')
            system = ('Answer questions strictly from the provided context. '
                      'If the answer is not in the context, say so clearly.')
            prompt = f'Context:\n{context}\n\nQuestion: {question}\n\nAnswer:'
            answer = self.llm.chat(prompt, system=system)

        total_tokens = estimate_tokens(prompt) + estimate_tokens(answer)
        print(f'  ✓ Done in {t.elapsed:.2f}s | ~{total_tokens:,} tokens')

        return QueryResult(
            pipeline='traditional', query=question,
            answer=answer, retrieved_context=context,
            latency_seconds=t.elapsed,
            estimated_tokens_used=total_tokens,
            metadata={'chunk_ids': [c.chunk_id for c in retrieved], 'top_k': self.top_k}
        )

    def show_chunks(self, n=3):
        print(f'\nFirst {n} chunks:')
        for c in self.vector_store._chunks[:n]:
            print(f'\n─ Chunk {c.chunk_id} (chars {c.start_char}–{c.end_char}):')
            print(c.text[:300])


print('✓ TraditionalRAGPipeline defined')

✓ TraditionalRAGPipeline defined


## Cell 8 — Page Index RAG: TOC Tree data structure

The core idea: instead of chunks, we build a tree of nodes where each node has
a title, a summary, and a pointer (page range) into the original document.

In [27]:
# ════════════════════════════════════════════════════════════
# PAGE INDEX RAG — TOC NODE DATA STRUCTURE
# ════════════════════════════════════════════════════════════

@dataclass
class TOCNode:
    """
    A single node in the Table of Contents tree.

    node_id    : unique identifier — also a pointer to source pages
    title      : short section heading
    summary    : one-sentence description of content
    start_page : first page (0-based) of this section
    end_page   : last page (inclusive) of this section
    children   : sub-nodes (sub-sections)
    tags       : semantic labels
    """
    node_id: str
    title: str
    summary: str
    start_page: int
    end_page: int
    children: list = field(default_factory=list)
    tags: list     = field(default_factory=list)

    def flat_repr(self, depth=0) -> str:
        indent = '  ' * depth
        lines = [f'{indent}[{self.node_id}] {self.title} '
                 f'(pages {self.start_page}–{self.end_page})']
        lines.append(f'{indent}    → {self.summary[:90]}')
        if self.tags:
            lines.append(f'{indent}    tags: {self.tags}')
        for child in self.children:
            lines.append(child.flat_repr(depth + 1))
        return '\n'.join(lines)

    def to_dict(self):
        return {
            'node_id': self.node_id, 'title': self.title,
            'summary': self.summary, 'start_page': self.start_page,
            'end_page': self.end_page, 'tags': self.tags,
            'children': [c.to_dict() for c in self.children]
        }

    @classmethod
    def from_dict(cls, d):
        return cls(
            node_id=d['node_id'], title=d['title'],
            summary=d['summary'], start_page=d['start_page'],
            end_page=d['end_page'], tags=d.get('tags', []),
            children=[cls.from_dict(c) for c in d.get('children', [])]
        )


print('✓ TOCNode defined')

✓ TOCNode defined


## Cell 9 — Page Index RAG: TOC Tree builder

This is where Page Index RAG differs fundamentally from traditional RAG.  
Instead of an embedding model, an **LLM reads the document** and extracts a structured tree.

In [28]:
# ════════════════════════════════════════════════════════════
# PAGE INDEX RAG — TOC TREE BUILDER
# ════════════════════════════════════════════════════════════

class TOCTreeBuilder:
    """
    Reads document pages in batches using an LLM and builds
    a hierarchical Table of Contents tree.

    The LLM reasons about:
      - Where sections begin and end
      - What each section is about (summary)
      - How sections relate hierarchically

    ⚠ Cost note: makes one LLM call per batch of pages.
    With batch_size=3 and a 15-page doc → 5 LLM calls.
    Save the tree JSON after indexing to avoid re-indexing.
    """

    def __init__(self, llm: LLMClient, batch_size=BATCH_SIZE):
        self.llm = llm
        self.batch_size = batch_size

    def build(self, pages: list) -> list:
        """Process all pages and return a list of root TOCNodes."""
        print(f'  Processing {len(pages)} pages, batch_size={self.batch_size}')
        all_nodes, node_counter = [], 0

        for start in range(0, len(pages), self.batch_size):
            end_idx = min(start + self.batch_size, len(pages))
            batch = pages[start:end_idx]
            print(f'  → Batch pages {start}–{end_idx-1}...', end=' ', flush=True)
            nodes = self._extract_nodes(batch, start, node_counter)
            node_counter += len(nodes)
            all_nodes.extend(nodes)
            print(f'{len(nodes)} node(s) found')

        return self._add_root(all_nodes, len(pages))

    def _extract_nodes(self, pages: list, page_offset: int, id_start: int) -> list:
        """Send a batch to the LLM and parse the returned JSON nodes."""
        pages_text = ''.join(
            f'\n--- PAGE {page_offset+i} ---\n{page}\n'
            for i, page in enumerate(pages)
        )

        prompt = f"""Analyze these document pages and identify distinct sections.

{pages_text}

Output a JSON array. Each element must have:
- "title": short heading (5-10 words)
- "summary": one sentence describing this section's content
- "start_page": integer page number where section starts
- "end_page": integer page number where section ends
- "tags": array of 1-3 topic tags
- "children": array of sub-sections (same structure) or []

Rules: identify natural section boundaries; keep summaries specific.
Output ONLY the JSON array, no markdown, no explanation."""

        raw = self.llm.chat(
            prompt,
            system='You are a document structure analyst. Output only valid JSON.'
        ).strip()

        # Strip markdown code fences if present
        if raw.startswith('```'):
            parts = raw.split('```')
            raw = parts[1][4:] if parts[1].startswith('json') else parts[1]
        raw = raw.strip()

        try:
            data = json.loads(raw)
        except json.JSONDecodeError as e:
            print(f'\n  ⚠ JSON parse error: {e} — skipping batch')
            return []

        nodes = []
        for i, item in enumerate(data):
            node = TOCNode(
                node_id   = f'node_{id_start+i:03d}',
                title     = item.get('title', 'Untitled'),
                summary   = item.get('summary', ''),
                start_page= item.get('start_page', page_offset),
                end_page  = item.get('end_page', page_offset + len(pages) - 1),
                tags      = item.get('tags', []),
                children  = [
                    TOCNode(
                        node_id=f'node_{id_start+i:03d}_c{j}',
                        title=c.get('title', ''), summary=c.get('summary', ''),
                        start_page=c.get('start_page', page_offset),
                        end_page=c.get('end_page', page_offset),
                        tags=c.get('tags', [])
                    )
                    for j, c in enumerate(item.get('children', []))
                ]
            )
            nodes.append(node)
        return nodes

    def _add_root(self, nodes: list, total_pages: int) -> list:
        """Create a single root node summarising the whole document."""
        toc = '\n'.join(f'- {n.title}: {n.summary[:70]}' for n in nodes)
        summary = self.llm.chat(
            f'Given these sections, write one sentence summarising the entire document:\n{toc}',
            system='Output only the summary sentence.'
        ).strip()
        root = TOCNode(
            node_id='root', title='Document root', summary=summary,
            start_page=0, end_page=total_pages - 1, children=nodes
        )
        return [root]


print('✓ TOCTreeBuilder defined')

✓ TOCTreeBuilder defined


## Cell 10 — Page Index RAG: Tree traversal retriever

The LLM reads the compact TOC tree and decides which nodes are relevant —  
then fetches only those pages from the source document.

In [29]:
# ════════════════════════════════════════════════════════════
# PAGE INDEX RAG — TREE TRAVERSAL RETRIEVER
# ════════════════════════════════════════════════════════════

class TreeTraversalRetriever:
    """
    Uses an LLM to navigate the TOC tree and identify which nodes
    are relevant to the user's query. No embeddings used here.

    This is the key step that replaces vector similarity search.
    Instead of computing distances between numbers, the LLM reads
    node titles and summaries and reasons about relevance.
    """

    def __init__(self, llm: LLMClient):
        self.llm = llm

    def retrieve(self, query: str, roots: list, pages: list):
        """
        1. Serialize the TOC tree
        2. Ask LLM to identify relevant node_ids
        3. Collect those nodes and fetch their source pages

        Returns: (relevant_nodes, source_text)
        """
        tree_repr = '\n'.join(r.flat_repr() for r in roots)

        prompt = f"""Navigate this document TOC tree to answer the question.

QUESTION: {query}

TREE:
{tree_repr}

Task:
1. Read node titles and summaries.
2. Select node_ids that are DIRECTLY relevant to answering the question.
3. Prune irrelevant branches entirely.

Output a JSON object:
{{"relevant_node_ids": ["node_001", ...], "reasoning": "one sentence"}}

Output ONLY the JSON object."""

        raw = self.llm.chat(
            prompt,
            system='You are a document navigator. Output only valid JSON.'
        ).strip()

        if raw.startswith('```'):
            parts = raw.split('```')
            raw = parts[1][4:] if parts[1].startswith('json') else parts[1]
        raw = raw.strip()

        try:
            result = json.loads(raw)
            relevant_ids = set(result.get('relevant_node_ids', []))
            reasoning    = result.get('reasoning', '')
        except json.JSONDecodeError:
            print('  ⚠ Parse error in traversal — using root')
            relevant_ids = {'root'}
            reasoning    = 'Fallback'

        print(f'  Reasoning : {reasoning}')
        print(f'  Selected  : {relevant_ids}')

        relevant_nodes = self._collect(roots, relevant_ids)
        source_text    = self._fetch_pages(relevant_nodes, pages)
        return relevant_nodes, source_text

    def _collect(self, roots: list, ids: set) -> list:
        """BFS to collect all nodes whose node_id is in ids."""
        found, queue = [], list(roots)
        while queue:
            node = queue.pop(0)
            if node.node_id in ids:
                found.append(node)
            queue.extend(node.children)
        return found

    def _fetch_pages(self, nodes: list, pages: list) -> str:
        """Fetch the exact source pages pointed to by each node."""
        seen, parts = set(), []
        for node in nodes:
            for idx in range(node.start_page, node.end_page + 1):
                if idx not in seen and idx < len(pages):
                    seen.add(idx)
                    parts.append(f'[Page {idx} — {node.title}]\n{pages[idx]}')
        return '\n\n'.join(parts)


print('✓ TreeTraversalRetriever defined')

✓ TreeTraversalRetriever defined


## Cell 11 — Page Index RAG: Full pipeline class

In [30]:
# ════════════════════════════════════════════════════════════
# PAGE INDEX RAG — FULL PIPELINE
# ════════════════════════════════════════════════════════════

class PageIndexRAGPipeline:
    """
    Full Page Index (Vectorless) RAG pipeline.

    Usage:
        pipeline = PageIndexRAGPipeline()
        pipeline.index(DOCUMENT_TEXT)
        pipeline.save_tree('tree.json')   # optional — skip re-indexing next time
        result = pipeline.query('What is reinforcement learning?')
        result.summary()
    """

    def __init__(self, chars_per_page=CHARS_PER_PAGE, batch_size=BATCH_SIZE):
        self.chars_per_page = chars_per_page
        self.llm        = LLMClient()
        self.builder    = TOCTreeBuilder(self.llm, batch_size)
        self.retriever  = TreeTraversalRetriever(self.llm)
        self._pages     = []
        self._roots     = []
        self._indexed   = False

    # ── INDEXING PHASE ─────────────────────────────────────

    def index(self, text: str):
        """
        INDEXING PHASE
        Step 1: Split document into pages
        Step 2: LLM reads pages and builds TOC tree

        ⚠ This makes multiple LLM calls — costs more than traditional
           RAG indexing, but only happens once. Save the tree after.
        """
        print('\n' + '='*52)
        print('  INDEXING PHASE — Page Index RAG')
        print('='*52)

        print(f'\n[1/2] Splitting into pages (~{self.chars_per_page} chars each)...')
        self._pages = split_into_pages(text, self.chars_per_page)
        print(f'  ✓ {len(self._pages)} pages')

        print('\n[2/2] Building TOC tree with LLM reasoning...')
        print('  (Multiple LLM calls — this is the cost trade-off)\n')
        self._roots = self.builder.build(self._pages)

        self._indexed = True
        print('\n  ✓ TOC tree built!')
        print('\n  Tree structure:')
        for root in self._roots:
            print(root.flat_repr())

    def save_tree(self, path='toc_tree.json'):
        """Save tree to JSON so you can reload without re-indexing."""
        with open(path, 'w') as f:
            json.dump([r.to_dict() for r in self._roots], f, indent=2)
        print(f'  ✓ Tree saved → {path}')

    def load_tree(self, path: str, text: str):
        """Load a previously saved tree — skips re-indexing."""
        with open(path) as f:
            data = json.load(f)
        self._roots   = [TOCNode.from_dict(d) for d in data]
        self._pages   = split_into_pages(text, self.chars_per_page)
        self._indexed = True
        print(f'  ✓ Tree loaded from {path}')

    def show_tree(self):
        for root in self._roots:
            print(root.flat_repr())

    # ── QUERY PHASE ────────────────────────────────────────

    def query(self, question: str) -> QueryResult:
        """
        QUERY PHASE
        Step 1: LLM traverses TOC tree (no embeddings!)
        Step 2: Fetch source pages via node_id pointers
        Step 3: LLM generates answer from those exact pages
        """
        if not self._indexed:
            raise RuntimeError('Call .index(text) first.')

        print('\n' + '='*52)
        print('  QUERY PHASE — Page Index RAG')
        print('='*52)
        print(f'  Question: {question}')

        with Timer() as t:
            print('\n[1/2] Traversing TOC tree...')
            nodes, source = self.retriever.retrieve(question, self._roots, self._pages)

            if not source:
                source = self._pages[0] if self._pages else ''

            ctx_tokens = estimate_tokens(source)
            print(f'  ✓ {len(nodes)} node(s) matched | ~{ctx_tokens} context tokens')

            print('\n[2/2] Generating answer...')
            system = ('Answer strictly from the document context provided. '
                      'Be specific. If not in context, say so.')
            prompt = f'Document context:\n{source}\n\nQuestion: {question}\n\nAnswer:'
            answer = self.llm.chat(prompt, system=system)

        total_tokens = estimate_tokens(prompt) + estimate_tokens(answer)
        print(f'  ✓ Done in {t.elapsed:.2f}s | ~{total_tokens:,} tokens')

        return QueryResult(
            pipeline='pageindex', query=question,
            answer=answer, retrieved_context=source,
            latency_seconds=t.elapsed,
            estimated_tokens_used=total_tokens,
            metadata={
                'nodes': [n.node_id for n in nodes],
                'titles': [n.title for n in nodes],
                'pages_fetched': list({p for n in nodes
                                       for p in range(n.start_page, n.end_page+1)})
            }
        )


print('✓ PageIndexRAGPipeline defined')

✓ PageIndexRAGPipeline defined


## Cell 12 — Index both pipelines

Run this once. After this cell both pipelines are ready to answer queries.

**Save the Page Index tree after indexing** — it avoids paying for re-indexing every session.

In [31]:
# ── Traditional RAG — index ─────────────────────────────────────────────
trad = TraditionalRAGPipeline()
trad.index(DOCUMENT_TEXT)

# Inspect chunks
trad.show_chunks(n=2)


  INDEXING PHASE — Traditional RAG

[1/3] Chunking document...
  ✓ 14 chunks created | avg size: 771 chars
  Sample (chunk 0): 'ARTIFICIAL INTELLIGENCE IN MODERN APPLICATIONS — A Comprehensive Guide\n\nCHAPTER 1: INTRODUCTION TO AI\n\nArtificial Intelligence (AI) refers to the simulation of human intelligence p'…

[2/3] Embedding 14 chunks...
  ✓ Embedding dimensions: 1024

[3/3] Storing in vector store...
  ✓ Stored 14 chunks (total: 14)

  ✓ Indexing complete!

First 2 chunks:

─ Chunk 0 (chars 0–800):
ARTIFICIAL INTELLIGENCE IN MODERN APPLICATIONS — A Comprehensive Guide

CHAPTER 1: INTRODUCTION TO AI

Artificial Intelligence (AI) refers to the simulation of human intelligence processes by computer systems.
These processes include learning, reasoning, and self-correction. The term was first coine

─ Chunk 1 (chars 700–1500):
.

Modern AI is divided into Narrow AI (ANI), which performs specific tasks like facial recognition, and
General AI (AGI), the concept of machines that can per

In [32]:
# ── Page Index RAG — index ──────────────────────────────────────────────
pi = PageIndexRAGPipeline()
pi.index(DOCUMENT_TEXT)

# Save tree — next session: pi.load_tree('toc_tree.json', DOCUMENT_TEXT)
pi.save_tree('toc_tree.json')


  INDEXING PHASE — Page Index RAG

[1/2] Splitting into pages (~2000 chars each)...
  ✓ 5 pages

[2/2] Building TOC tree with LLM reasoning...
  (Multiple LLM calls — this is the cost trade-off)

  Processing 5 pages, batch_size=3
  → Batch pages 0–2... 4 node(s) found
  → Batch pages 3–4... 
  ⚠ JSON parse error: Expecting value: line 4 column 16 (char 51) — skipping batch
0 node(s) found

  ✓ TOC tree built!

  Tree structure:
[root] Document root (pages 0–4)
    → The document provides an overview of Artificial Intelligence, covering its introduction, M
  [node_000] Introduction to AI (pages 0–0)
      → Artificial Intelligence refers to the simulation of human intelligence processes by comput
      tags: ['AI', 'Introduction']
  [node_001] Machine Learning Fundamentals (pages 0–1)
      → Machine Learning provides systems the ability to learn and improve from experience without
      tags: ['Machine Learning', 'AI']
    [node_001_c0] Supervised Learning (pages 0–0)
        → Super

## Cell 13 — Query both pipelines and compare

Change `QUERY` to any question about the document.

In [33]:
QUERY = 'What is reinforcement learning and what are its key concepts?'

r_trad = trad.query(QUERY)
r_pi   = pi.query(QUERY)

print_comparison(r_trad, r_pi)


  QUERY PHASE — Traditional RAG
  Question: What is reinforcement learning and what are its key concepts?

[1/3] Embedding query...

[2/3] Similarity search (top-5)...

  Top-5 similarity scores:
    [0.5528] Chunk 3: 't to make decisions by interacting with an environment.'…
    [0.4714] Chunk 2: 'ession, Logistic Regression, Decision Trees, Random For'…
    [0.3808] Chunk 4: 'rning uses neural networks with many layers to learn fr'…
    [0.3564] Chunk 8: 'ntion (attention applied multiple times in parallel), P'…
    [0.3470] Chunk 5: 'images. They use convolutional layers that apply filter'…

[3/3] Generating answer (~1006 context tokens)...
  ✓ Done in 5.52s | ~1,193 tokens

  QUERY PHASE — Page Index RAG
  Question: What is reinforcement learning and what are its key concepts?

[1/2] Traversing TOC tree...
  Reasoning : The question about reinforcement learning and its key concepts is directly related to the Machine Learning Fundamentals and Reinforcement Learning nodes.
  Selecte

In [34]:
# Inspect the retrieved context for each pipeline
print('═'*60)
print('TRADITIONAL RAG — context passed to LLM:')
print('═'*60)
print(r_trad.retrieved_context[:1200])
print('\n...(truncated)\n')

print('═'*60)
print('PAGE INDEX RAG — source pages fetched:')
print('═'*60)
print(r_pi.retrieved_context[:1200])
print('\n...(truncated)\n')

print('Page Index node metadata:')
for k, v in r_pi.metadata.items():
    print(f'  {k}: {v}')

════════════════════════════════════════════════════════════
TRADITIONAL RAG — context passed to LLM:
════════════════════════════════════════════════════════════
t to make decisions by interacting with an environment.
The agent receives rewards for correct actions and penalties for incorrect ones, aiming to maximize
cumulative reward over time.

Key RL concepts: Agent (the learner), Environment (what the agent interacts with), State (current
situation), Action (what the agent can do), Reward (feedback signal), Policy (the agent's strategy),
and Value Function (expected cumulative reward from a state).

Notable RL algorithms: Q-Learning, Deep Q-Network (DQN), Proximal Policy Optimization (PPO), and
Actor-Critic methods. RL has been applied to games (AlphaGo), robotics, and recommendation systems.

CHAPTER 3: DEEP LEARNING AND NEURAL NETWORKS

Deep learning uses neural networks with many layers to learn from large amounts of data. It has driven
remar

---

ession, Logistic
Regression, D

## Cell 14 — Experiment A: Context split problem

Shrink `chunk_size` to see how traditional RAG cuts concepts in half.

In [35]:
# Create a chunker with small, zero-overlap windows
demo_chunker_no_overlap  = TextChunker(chunk_size=250, overlap=0)
demo_chunker_with_overlap = TextChunker(chunk_size=250, overlap=60)

no_overlap_chunks   = demo_chunker_no_overlap.chunk(DOCUMENT_TEXT)
with_overlap_chunks = demo_chunker_with_overlap.chunk(DOCUMENT_TEXT)

print('='*60)
print('CHUNK BOUNDARY DEMO — same 3 consecutive chunks')
print('='*60)

for i in [4, 5, 6]:
    print(f'\n── No overlap | Chunk {i}:')
    print(repr(no_overlap_chunks[i].text))

    print(f'\n── With overlap=60 | Chunk {i}:')
    print(repr(with_overlap_chunks[i].text))

print('\n→ Notice: overlap=60 shares 60 chars with the previous chunk.')
print('→ It reduces hard cuts but does NOT eliminate them — just shifts them.')

CHUNK BOUNDARY DEMO — same 3 consecutive chunks

── No overlap | Chunk 4:
'ems the ability to learn and improve from\nexperience without being explicitly programmed. ML focuses on developing programs that can access data\nand learn from it autonomously.\n\nSECTION 2.1 — SUPERVISED LEARNING\n\nIn supervised learning, the algorithm'

── With overlap=60 | Chunk 4:
'specific tasks like facial recognition, and\nGeneral AI (AGI), the concept of machines that can perform any intellectual task a human can.\n\nCHAPTER 2: MACHINE LEARNING FUNDAMENTALS\n\nMachine Learning (ML) is a subset of AI that provides systems the ab'

── No overlap | Chunk 5:
'is trained on labeled data. The model learns a mapping from inputs\nto outputs based on example input-output pairs. Key algorithms include Linear Regression, Logistic\nRegression, Decision Trees, Random Forest, Support Vector Machines (SVM), and Neura'

── With overlap=60 | Chunk 5:
'Learning (ML) is a subset of AI that provides systems the ability t

## Cell 15 — Experiment B: Vague vs precise query

Traditional RAG uses keyword matching. A vague query can miss content
that Page Index finds via structural reasoning.

In [36]:
# Precise — traditional RAG should do fine
precise = 'What are LSTM and GRU and how do they solve the vanishing gradient problem?'
rt_p = trad.query(precise)
rp_p = pi.query(precise)
print_comparison(rt_p, rp_p)


  QUERY PHASE — Traditional RAG
  Question: What are LSTM and GRU and how do they solve the vanishing gradient problem?

[1/3] Embedding query...

[2/3] Similarity search (top-5)...

  Top-5 similarity scores:
    [0.4647] Chunk 5: 'images. They use convolutional layers that apply filter'…
    [0.4634] Chunk 6: 'and speech recognition.  RNN variants: LSTM (Long Short'…
    [0.3527] Chunk 7: 'from co-occurrence), GloVe (Global Vectors), FastText ('…
    [0.3518] Chunk 3: 't to make decisions by interacting with an environment.'…
    [0.3407] Chunk 4: 'rning uses neural networks with many layers to learn fr'…

[3/3] Generating answer (~1006 context tokens)...
  ✓ Done in 11.81s | ~1,167 tokens

  QUERY PHASE — Page Index RAG
  Question: What are LSTM and GRU and how do they solve the vanishing gradient problem?

[1/2] Traversing TOC tree...
  Reasoning : Recurrent Neural Networks, which include LSTM and GRU, are directly relevant to solving the vanishing gradient problem in sequential d

In [37]:
# Vague — no technical keywords, traditional RAG may miss the relevant chunk
vague = 'How do machines deal with the problem of forgetting earlier information?'
rt_v = trad.query(vague)
rp_v = pi.query(vague)
print_comparison(rt_v, rp_v)


  QUERY PHASE — Traditional RAG
  Question: How do machines deal with the problem of forgetting earlier information?

[1/3] Embedding query...

[2/3] Similarity search (top-5)...

  Top-5 similarity scores:
    [0.3293] Chunk 8: 'ntion (attention applied multiple times in parallel), P'…
    [0.3243] Chunk 3: 't to make decisions by interacting with an environment.'…
    [0.3103] Chunk 6: 'and speech recognition.  RNN variants: LSTM (Long Short'…
    [0.3056] Chunk 1: '.  Modern AI is divided into Narrow AI (ANI), which per'…
    [0.2890] Chunk 0: 'ARTIFICIAL INTELLIGENCE IN MODERN APPLICATIONS — A Comp'…

[3/3] Generating answer (~1006 context tokens)...
  ✓ Done in 5.90s | ~1,108 tokens

  QUERY PHASE — Page Index RAG
  Question: How do machines deal with the problem of forgetting earlier information?

[1/2] Traversing TOC tree...
  Reasoning : Machines deal with the problem of forgetting earlier information through the use of Recurrent Neural Networks, which are designed for sequent

## Cell 16 — Experiment C: Cross-reference query

This question spans two different chapters — CNNs from chapter 3 AND
healthcare from chapter 6. Traditional RAG chunks rarely link these.
Page Index can select nodes from multiple branches.

In [38]:
cross_ref = ('How are convolutional neural networks used in healthcare, '
             'and why are they chosen over other network types for this?')

rt_c = trad.query(cross_ref)
rp_c = pi.query(cross_ref)
print_comparison(rt_c, rp_c)

print('\nPage Index nodes selected for this cross-reference query:')
for title in rp_c.metadata.get('titles', []):
    print(f'  → {title}')


  QUERY PHASE — Traditional RAG
  Question: How are convolutional neural networks used in healthcare, and why are they chosen over other network types for this?

[1/3] Embedding query...

[2/3] Similarity search (top-5)...

  Top-5 similarity scores:
    [0.4632] Chunk 5: 'images. They use convolutional layers that apply filter'…
    [0.4561] Chunk 10: 'on divides images into meaningful regions at pixel leve'…
    [0.4220] Chunk 4: 'rning uses neural networks with many layers to learn fr'…
    [0.3648] Chunk 8: 'ntion (attention applied multiple times in parallel), P'…
    [0.3621] Chunk 11: 'traditional methods. The FDA has approved over 500 AI-e'…

[3/3] Generating answer (~1006 context tokens)...
  ✓ Done in 18.57s | ~1,167 tokens

  QUERY PHASE — Page Index RAG
  Question: How are convolutional neural networks used in healthcare, and why are they chosen over other network types for this?

[1/2] Traversing TOC tree...
  Reasoning : Convolutional Neural Networks are directly relevan

## Cell 17 — Batch comparison across multiple queries

In [39]:
test_queries = [
    'What is the difference between supervised and unsupervised learning?',
    'How does the transformer attention mechanism work?',
    'What ethical concerns does AI raise?',
    'How is AI used in autonomous vehicles?',
]

results = []
for q in test_queries:
    print(f'\nQuerying: {q[:55]}...')
    rt = trad.query(q)
    rp = pi.query(q)
    results.append((rt, rp))

print('\n✓ All queries complete')
print(f'\n{"Query":<50} {"Trad(s)":<10} {"PI(s)":<10} {"Trad tok":<12} {"PI tok"}')
print('─' * 95)
for rt, rp in results:
    print(f'{rt.query[:49]:<50} {rt.latency_seconds:<10.2f} {rp.latency_seconds:<10.2f} '
          f'{rt.estimated_tokens_used:<12,} {rp.estimated_tokens_used:,}')


Querying: What is the difference between supervised and unsupervi...

  QUERY PHASE — Traditional RAG
  Question: What is the difference between supervised and unsupervised learning?

[1/3] Embedding query...

[2/3] Similarity search (top-5)...

  Top-5 similarity scores:
    [0.4562] Chunk 2: 'ession, Logistic Regression, Decision Trees, Random For'…
    [0.3588] Chunk 1: '.  Modern AI is divided into Narrow AI (ANI), which per'…
    [0.3315] Chunk 5: 'images. They use convolutional layers that apply filter'…
    [0.3294] Chunk 4: 'rning uses neural networks with many layers to learn fr'…
    [0.3122] Chunk 6: 'and speech recognition.  RNN variants: LSTM (Long Short'…

[3/3] Generating answer (~1006 context tokens)...
  ✓ Done in 11.50s | ~1,121 tokens

  QUERY PHASE — Page Index RAG
  Question: What is the difference between supervised and unsupervised learning?

[1/2] Traversing TOC tree...
  Reasoning : The difference between supervised and unsupervised learning can be found in th

The results expose both pipelines' real strengths and weaknesses in practice.

**Traditional RAG** retrieved consistently across all 4 queries but with low similarity scores (0.29–0.57), meaning it pulled partially irrelevant chunks every time — it found something but not always the right thing.

**Page Index RAG** showed smarter selection when the tree had the right nodes — it pinpointed exactly 1 node for the transformer question (634 tokens vs 1,149) saving half the context. But it completely failed on "ethical concerns" returning 0 nodes, because the TOC tree didn't label that section clearly enough during indexing — demonstrating the core weakness: a bad tree means a failed query.

**The autonomous vehicles query** revealed Page Index's over-retrieval problem — it selected 8 nodes and 1,532 tokens of context because the LLM reasoned that ML and deep learning are related to autonomous vehicles, pulling in irrelevant sections. More nodes does not mean better answers.

## Cell 18 — Try your own PDF

Upload any PDF and run both pipelines on it.

In [40]:
from google.colab import files

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(f'Uploaded: {'/content/vectorless_rag_explained.pdf'}')

your_doc = load_document(pdf_path)
print(f'Characters: {len(your_doc):,}')
print(f'Preview: {your_doc[:300]}')

Saving vectorless_rag_explained.pdf to vectorless_rag_explained (1).pdf
Uploaded: /content/vectorless_rag_explained.pdf
Characters: 1,170
Preview: Vectorless RAG (Page Index) – Complete Explanation
Intro & Overview
So hey everyone, welcome back! Welcome back to another exciting video, and in this particular video, I have
something very exciting for you. In this video, we will talk about Vectorless RAG. Yes, that means this is a new
shift—a new


In [41]:
your_trad = TraditionalRAGPipeline()
your_trad.index(your_doc)

your_pi = PageIndexRAGPipeline()
your_pi.index(your_doc)
your_pi.save_tree('your_toc_tree.json')


  INDEXING PHASE — Traditional RAG

[1/3] Chunking document...
  ✓ 2 chunks created | avg size: 634 chars
  Sample (chunk 0): 'Vectorless RAG (Page Index) – Complete Explanation\nIntro & Overview\nSo hey everyone, welcome back! Welcome back to another exciting video, and in this particular video, I have\nsome'…

[2/3] Embedding 2 chunks...
  ✓ Embedding dimensions: 1024

[3/3] Storing in vector store...
  ✓ Stored 2 chunks (total: 2)

  ✓ Indexing complete!

  INDEXING PHASE — Page Index RAG

[1/2] Splitting into pages (~2000 chars each)...
  ✓ 1 pages

[2/2] Building TOC tree with LLM reasoning...
  (Multiple LLM calls — this is the cost trade-off)

  Processing 1 pages, batch_size=3
  → Batch pages 0–0... 1 node(s) found

  ✓ TOC tree built!

  Tree structure:
[root] Document root (pages 0–0)
    → The document provides an introduction to Vectorless RAG, highlighting its benefits and fea
  [node_000] Introduction to Vectorless RAG (pages 0–0)
      → The video introduces Vectorless 

In [46]:
### Pasted text of of doc
# Paste the full transcript text here directly
your_doc = """
Vectorless RAG (Page Index) – Complete Explanation

Intro & Overview
So hey everyone, welcome back!...

[
Intro & Overview
So hey everyone, welcome back! Welcome back to another exciting video, and in this particular
video, I have something very exciting for you. In this video, we will talk about Vectorless RAG. Yes,
that means this is a new shift—a new way and a new pipeline of doing RAG, but without any
vectors.
Don't worry, within this video, let me first explain to you what traditional RAG is, what vector RAG
is, what the problems with vector RAG are, and how this new approach—Vectorless RAG (also
known as Page Index)—solves a major problem and utilizes reasoning models to improve
document retrieval. With that, let's start the video.
All right. Before jumping into Vectorless RAG, let's first understand what Traditional RAG is, okay?
And even before that, what is RAG? RAG basically stands for Retrieval-Augmented Generation.
The problem statement here is very simple. For now, let's forget about Vectorless RAG; let's not
worry about it. Let's say that in a traditional application, you have a lot of documents. I will take
a PDF file here as an example. Let's say this is my PDF file, and you have a lot of PDF files or a lot
of pages within a PDF file, and the user wants to run some Q&A over it using AI, okay?
That means I have these three pages—it can be 3 pages or it can be 300 pages—and I need to do
some Q&A on them. Simply put, the naive solution would be: what will the user do? They will
give you a query, right? The user is going to give you a query, which we also call a prompt. Let's
say this is my query. What you can do in an LLM model—you can use any Large Language Model,
it can be GPT, OpenAI, Anthropic, any model; let's assume this is my model. So, what you can
basically do is pass all these documents into it. You take all your documents and content, put
them into the prompt, and you put the user query into the prompt as well. Then, this LLM can
generate some kind of output.
This is one naive solution that always works, where you provide all the document content and the
user query, execute a generation via an LLM call, and get an output. But as you can see in the
diagram, things don't work that simply. There are a lot of problems with this particular approach.
Core Problems with the Naive LLM Approach
1. Large Context Windows
The first problem that arises is the very large context. Because these files can be massive, they
can contain a huge amount of content, and LLMs have a limited context window, okay? That
means if you stuff too much content into it, the LLM can fail because the context window is
restricted. You cannot ingest 3,000 pages at once. If it's a couple of pages, you can pass them;
that's completely fine. But if your PDF file has even 100 pages, there is a high probability that your
LLM will fail.
Let's assume that a year or two from now, LLM context windows increase significantly, and you
can even ingest 3,000 pages.
2. Information Overload & Hallucination
The second problem that comes up here is dealing with too much context, which causes the LLM
to start hallucinating. If you have a 3,000-page PDF and ingest the entire thing because your LLM
now accommodates a massive context, the quality of the output will not be good. Because there
is too much context, the model loses focus. You have essentially ingested an entire book into the
LLM, so the focus is lost. The answers you get might be generic rather than a focused response.
That is a major problem.
3. Financial Cost
The third problem is the cost. Even if the user's query is highly specific and simple—for example,
only targeting page number 5—if you ingest 3,000 pages for every single LLM call, you obviously
have to pay a massive cost for the tokens. In the case of LLMs, everything is a token, and tokens
are expensive. Therefore, this is not an efficient solution, right? Why should I supply 3,000 pages
just for a single user query when it increases my costs and decreases the quality of the output?
This was a well-known bottleneck inside LLMs, and that is exactly where RAG enters the picture.
Again, my friends, I am not talking about vectors or vectorless approaches yet; we are simply
understanding the problem statement right now, okay?
The Traditional Vector RAG Pipeline
This is where the Traditional RAG system came in. A RAG system solves the problem of how to
ingest large documents into LLMs. Within traditional RAG, you have two phases:
1. The Indexing Phase
2. The Query Phase
Let's talk about the indexing phase first. What does the indexing phase say? The user will provide
you with some PDF files (it could be PDFs, Excel, Doc files, any kind of file, but for now, we will
stick to PDFs since it's the most common format).
The Indexing Phase:
• Chanking (Chunking): The first part of RAG is chunking. Chunking means that I can split
the text using a particular algorithm. A simple chunking method could be page-by-page
chunking. If I have 3,000 pages, I will create 3,000 chunks out of my PDF file, which works
perfectly fine.
• Paragraph Chunking: If you want to make the chunks smaller, you can do paragraph-byparagraph chunking. But there is a problem with that too: if a paragraph is exceptionally
large, you can still overshoot your target context window size.
• Fixed Window Chunking: Usually, developers opt for fixed window chunking. I will pick a
size here—say, 500 words. What I will do is create chunks across this entire PDF based on
blocks of 500 words. Let's say this is Chunk 1, Chunk 2, Chunk 3, and so on.
Now, what you basically do is convert these chunks into vectors using an embedding model. For
example, if you are using OpenAI, they provide specialized models for creating vectors; you don't
use simple LLM models here. Let me show you vector models. If we look at OpenAI, you can see
vector embeddings and their specific models like text-embedding-3-small or text-embedding-3-
large. You use these distinct models for embeddings. You take these chunks, call your embedding
model, and it returns an array of numbers. Because at the end of the day, what are vectors? They
are just an array of numbers.
(Just in case you want to understand what vector embeddings are in detail, I have a dedicated
video on that, and I highly recommend watching it. The links are in the description below.)
Next, you have to store these embeddings somewhere in a database. You cannot save these
vector embeddings inside traditional databases. There are specialized databases for vector
embeddings, such as Pinecone. Pinecone DB is a vector database. Similarly, you have ChromaDB,
Weaviate, Milvus, and Qdrant. There are many vector databases available. Even within
PostgreSQL, there is an extension called pgvector that turns it into a vector database. We save
these vectors inside our vector DB along with their corresponding text chunk. That means this
text belongs to this vector embedding. For however many chunks you created, you will save that
many vector embeddings into your database. That is the first part: Indexing. That's it. Take the
PDF file, chunk it, create its vectors, and save them to the database. Your indexing phase is done.
The Query Phase:
The second phase occurs when a user wants to chat or ask something about their PDF file. What
happens is your user comes in and provides a query. They say, "Hey, this is my query, please tell
me something about this based on my PDF file."
What you do first is take this query and, using the exact same embedding model, convert the
user's query into a vector embedding. It will generate some numbers—let's say the resulting
vector looks like [3, 2, 5, 6]. Now, you can search for similar numbers inside your database.
Remember our Pinecone database? You will query the database and perform a Vector Similarity
Search. You tell the database, "Hey, I have these numbers [3, 2, 5, 6], search and find matches." It
will identify which vector points lie closest to it.
Say the user asked a question about a car; wherever a car is discussed in our PDF, the vector
embeddings will align, and you will retrieve the relevant chunks. You also pass another parameter
here called Top-K, which dictates how many relevant chunks you want. If I set my Top-K to 5, it
will fetch the top 5 most relevant chunks. Not the whole PDF file—just those specific chunks. And
since we decided a chunk size might be 500 words or a single paragraph, you get Top 1, 2, 3, 4,
and 5 chunks. The original PDF file might have been 3,000 pages, but by smartly using the user's
query, you isolated only the relevant chunks. "Relevant" means the chunks where the content
aligns with the user's query.
Now, you take these chunks plus the user's original query, and you make a standard LLM call (this
can be GPT-4o, Claude Anthropic, whatever you prefer). You tell it, "Hey, here are the relevant
chunks, and here is the user's query. Now, perform a generation based on this." You will get a
result which you can then return to the user. This is how your traditional RAG system works by
utilizing vectors.
Structural Flaws in Vector RAG
Vector RAG works fine; it is heavily used across the industry today, almost every company uses it,
and it's a very traditional, established way of doing document RAG. But what is the biggest
problem it faces? The problem is chunking, because we don't have a solid semantic justification
for how we chunk text.
The Context Split Problem
Let's say you have paragraph 1, paragraph 2, paragraph 3, and paragraph 4. You blindly picked an
algorithm that splits text strictly at 500 words. If this is an entire page, you slice it up: grab the
first 500 words, then the next 500, and so on. The catch is that a vital piece of information might
start in one chunk and end in the next. Because you sliced it right down the middle, your context
is lost, your data is fractured.
Imagine opening a random paragraph—even the one from this Vectorless RAG article. Assume
it's a storybook. If my 500-character limit hits right in the middle of a sentence, the first chunk
stops there, and the second chunk picks up the rest. Logically, you need both parts to understand
the story, but because you chunked based on a static number, half the context stayed in one
chunk, and the other half went into the next. The paragraph is broken.
Secondly, what if three paragraphs collectively form a single storyline? It's rarely the case that
every single paragraph has its own completely isolated narrative. They often group together. But
if you do paragraph-by-paragraph chunking, they get split into independent units. Furthermore,
paragraph three might be tiny while paragraph two is massive. The chunking strategy we use has
no structural justification; we are doing it blindly. We need a way to build chunks semantically,
not through hardcoded constraints like "every paragraph" or "every 500 words." Hardcoded
chunking creates hard cuts in context.
The Cross-Reference Problem
Another problem happens with documents like legal docs. Legal documents are full of crossreferences. They might say something like, "As per rule/appendix 63.7.4 of Section A..." and then
continue their point. You can clearly see there is a reference to an entirely different page. The
reference is on page 4, but the actual rule being referenced might be stated on page 578 of that
same PDF. To generate an accurate answer, the model needs to read both pages simultaneously.
But chunking doesn't work that way. In vector embeddings, the system will look at the keywords
on page 4, pull that chunk, and completely miss page 578 because the text there doesn't look
keywords-similar to the initial user query. That is a massive failure point of chunking.
Over-Reliance on User Query Formulation
The third problem in vector RAG occurs during the vector similarity search. This search runs purely
on mathematical distance between numbers, meaning these numbers heavily rely on how well
the user phrases their question. You have zero control over how a user writes. If I write a highly
optimized prompt for my LLM using the exact keywords present in the PDF file, its vector
embedding will align seamlessly with the embeddings stored in Pinecone, yielding highly accurate
chunks.
But that doesn't happen every time. The ingested book might use highly technical terminology,
while the user asks a very vague, high-level question like, "How do I do this?" The vector
embedding generated for that vague question might never map close to the original document's
vector space because the user doesn't know the exact keywords used in the book. We are relying
entirely on the assumption that the user's query will be high-quality and match our document
embeddings. If the user's query is poor, we get irrelevant documents, and our LLM output
degrades.
These are the glaring issues with traditional RAG, and these problems are now being solved using
Vectorless RAG.
Introducing Vectorless RAG (Page Index)
Vectorless RAG, as the name implies, does not use vector embeddings at all. It still has two
phases—the Indexing phase and the Query phase—so the macro flow looks identical, but the
underlying mechanisms are completely transformed. There are no vectors, no Pinecone, no
embeddings, and not even any chunking.
Instead, you use a Reasoning Model. Over time, LLMs have grown significantly smarter, more
capable, and highly adept at logical reasoning. Vectorless RAG relies heavily on these reasoning
capabilities. Let's look at this documentation article showcasing a Sholay movie example.
This approach is called Page Index. The alternative name for Vectorless RAG is Page Index—how
to build RAG without vector embeddings or a vector DB. Reading through this document, you can
clearly see: "Page Index is a vectorless, reasoning-based retrieval-augmented generation (RAG)
framework." It is completely vectorless. Instead of relying on semantic similarity searches (the
vector searches I explained earlier), Page Index builds a Hierarchical Table of Contents (TOC) Tree.
This is where data structures come in handy! This is a crucial concept: a Hierarchical Table of
Contents.
The New Indexing Phase: Building the TOC Tree
In this indexing phase, you aren't creating vector embeddings or doing traditional chunking. What
you basically do is construct a TOC Tree. You've studied trees in data structures, right? A tree looks
like this: you have a root node, which branches out into multiple child nodes, and these nodes
are interconnected.
When you buy a book, it comes with an index. If you want to read something specific, you look at
the index, find the relevant section, and jump straight to it, correct? This is exactly what we need
to build. But how do we build this tree? You extract this TOC from the document by using a Large
Language Model to reason over its native layout and structure. The model analyzes the
document's inherent hierarchy. Later, during the query phase, the model navigates this tree to
generate precise answers.
To compare: Traditional RAG works strictly on text similarity, whereas Page Index works via
reasoning. It is heavily inspired by human behavior. If I hand you a massive, thick book and ask
you a specific question, how does your brain find the answer? It doesn't run a mathematical
vector search across all pages simultaneously; it mimics a Page Index.
This completely bypasses the legal document and contract cross-referencing issues I mentioned
earlier. Page Index structures the data before any search happens by building a hierarchical index.
The document comes in, you generate its hierarchical index, apply reasoning-based retrieval over
it, and output the answer.
The Sholay Movie Example
Let's see how this works using a book or script of the movie Sholay. You ask the LLM to go through
the document page-by-page and compile an index. What will that index tree look like?
• Root Node: A top-level node (which can be initialized as null or a base pointer) containing
a macro summary of what the entire Sholay movie is about.
• Child Nodes (Plots/Scenarios): Beneath the root, the LLM identifies key plots, shifts, and
scenarios. "Scenarios" or "Scene Headings" is the correct terminology here. The LLM
handles this structural detection automatically; reasoning models are highly capable of
this.
It maps out sections like: Life in Ramgarh, Gabbar's Reign, The Recruitment of Veeru and Jai, and
The Final Showdown. It captures the main headings and vital scenarios where plot twists occur or
specific stories conclude. There is no actual content text stored at these top layers—only
descriptive structural headings. The structural detection maps out scenes, character
introductions, act breaks, and major transitions.
Notice that there is no fixed chunk size here. We aren't cutting text blindly at 500 words. Based
purely on contextual reasoning, the LLM identifies boundaries where things shift—such as a new
character entering, a massive twist, an emotional sequence, or the ending. The LLM detects
exactly where context transforms, and based on that, it constructs the tree. Programmers don't
have to hardcode these rules; the LLM handles the tree construction.
You can also assign attributes or tags to nodes. For example, tagging story segments in blue, all
nodes related to the villain Gabbar in red, critical events in purple, and generic events in gold.
LLMs excel at categorization. You pass the document to the LLM, let it reason over it, and use that
reasoning to output a beautifully mapped hierarchical tree representing your document.
Node Data Structure
What data do we store inside each individual node of this tree?
1. Title: The name/heading of the node.
2. Node ID: A crucial unique ID that acts as a direct reference or pointer to the exact location
or page number in the original physical document where this text lives.
3. Summary: A concise summary of the content bounded by this node.
4. Child Nodes: An array containing references to its sub-nodes.
This is exactly how you map a tree structure in memory.
The Query Phase: Tree Traversal
Now, what happens when a user asks a query? For example, they ask: "Why did Thakur lose his
arms?"
To answer this, you do not need to feed the entire movie script to the LLM. We want to keep our
context size minimal. There are no embeddings generated, and no similarity searches executed.
Instead, using the user's question, the system performs a tree traversal (such as a Breadth-First
Search or similar traversal) directly over the TOC tree to isolate the relevant nodes.
We don't touch the original massive document text yet. We isolate our operations to the compact
Table of Contents tree we created. We instruct the LLM: "Hey, the user has asked this question.
Here is our document tree. Please traverse it, identify which nodes are contextually relevant to
this question, and fetch them along with their child nodes."
The LLM navigates the hierarchical map against the user's prompt, reading through the node titles
and summaries. Based on structural reasoning, it pulls the exact nodes that matter. Because your
tree is cleanly structured, it might select two or three highly relevant nodes from different
branches of the tree, ignoring everything else.
Once you have isolated these specific relevant nodes, you look up their Node ID pointers. You
then fetch the exact original text segments or pages from the source document. Since you also
have the summaries handy, you can factor them in too. The LLM uses its reasoning to determine
exactly which nodes are required, pulls their source content, and executes the final retrieval and
generation.
To visualize: the user's query comes in. The system evaluates the tree. It determines Node A is
highly relevant, but Node B is completely irrelevant. If a branch is irrelevant, the system prunes it
entirely and skips its children. It picks only the specific nodes that contextually fit the query,
obtaining a highly accurate subset of the tree. The summaries stored at each node are what allow
the LLM to make these routing decisions. Finally, it pulls the raw source text via the node pointers,
gives it to the LLM, and runs the final answer generation.
Everything is driven purely by the structural reasoning and intrinsic capabilities of the LLM. This
is the core operating mechanism behind Page Index.
Conclusion & Trade-offs
Page Index shifts the paradigm toward intelligent navigation and targeted extraction. This closely
mimics human reading patterns. When you want to find out why a specific event happened in a
novel, you don't flip through every single page randomly or skim every sentence simultaneously;
you flip to the specific chapter where that event took place. Page Index forces LLMs to behave in
that exact same structured manner.
There is an open-source Python SDK implementing this exact framework called Page Index. You
feed it a document, it automatically constructs the hierarchical tree, applies LLM reasoning over
the query via tree traversal, and outputs your answer. This is a relatively brand-new concept in
the AI landscape. To reiterate what a node looks like structurally: it holds a title, a unique node
ID, a start index, an end index (locating it precisely in the master file), a summary, and an array of
child nodes.
Because modern LLMs have become incredibly smart over time, we can leverage their advanced
reasoning models to manage this entire process. That is how Vectorless RAG comes into the
picture.
We recently migrated one of our production projects from a traditional vector-based RAG pipeline
to a Vectorless RAG architecture. Through that experience, we observed two main trade-offs you
must consider:
1. Financial Cost: Reasoning models consume significantly more tokens during the index
mapping and traversal phases, making them more expensive.
2. Latency (Speed): Because the model has to logically reason through nodes and perform
step-by-step tree traversal, it takes slightly more time for the LLM to yield the final output
compared to a fast vector lookup.
With Vectorless RAG, we are trading speed for unmatched accuracy. It's a trade-off we gladly
accepted for our use case. This is a very fresh and evolving area in the AI space, and as we know,
things change rapidly.
Let me know in the comments what you think about Vectorless RAG! What are your thoughts on
it, and how do you plan to use it? If you want me to create a step-by-step coding video
implementing a vectorless pipeline, let me know below—I'd be more than happy to do it since
we just built one and it is surprisingly easy to implement once you get the hang of it.
If you enjoyed this video, please make sure to like and subscribe. See you in the next one!

]
"""

# Now re-index both pipelines on the full text
your_trad = TraditionalRAGPipeline()
your_trad.index(your_doc)

your_pi = PageIndexRAGPipeline()
your_pi.index(your_doc)
your_pi.save_tree('your_toc_tree.json')



  INDEXING PHASE — Traditional RAG

[1/3] Chunking document...
  ✓ 33 chunks created | avg size: 785 chars
  Sample (chunk 0): 'Vectorless RAG (Page Index) – Complete Explanation\n\nIntro & Overview\nSo hey everyone, welcome back!...\n\n[\nIntro & Overview\nSo hey everyone, welcome back! Welcome back to another ex'…

[2/3] Embedding 33 chunks...
  ✓ Embedding dimensions: 1024

[3/3] Storing in vector store...
  ✓ Stored 33 chunks (total: 33)

  ✓ Indexing complete!

  INDEXING PHASE — Page Index RAG

[1/2] Splitting into pages (~2000 chars each)...
  ✓ 12 pages

[2/2] Building TOC tree with LLM reasoning...
  (Multiple LLM calls — this is the cost trade-off)

  Processing 12 pages, batch_size=3
  → Batch pages 0–2... 3 node(s) found
  → Batch pages 3–5... 4 node(s) found
  → Batch pages 6–8... 4 node(s) found
  → Batch pages 9–11... 4 node(s) found

  ✓ TOC tree built!

  Tree structure:
[root] Document root (pages 0–11)
    → The document introduces Vectorless RAG, a framework that d

In [47]:
YOUR_QUERY = 'What are the problems with chunking in vector RAG?'

yt = your_trad.query(YOUR_QUERY)
yp = your_pi.query(YOUR_QUERY)
print_comparison(yt, yp)


  QUERY PHASE — Traditional RAG
  Question: What are the problems with chunking in vector RAG?

[1/3] Embedding query...

[2/3] Similarity search (top-5)...

  Top-5 similarity scores:
    [0.6841] Chunk 16: 'n clearly see there is a reference to an entirely diffe'…
    [0.6174] Chunk 13: 'uctural Flaws in Vector RAG Vector RAG works fine; it i'…
    [0.5282] Chunk 14: 'xt. Because you sliced it right down the middle, your c'…
    [0.5249] Chunk 12: 'unk size might be 500 words or a single paragraph, you '…
    [0.5171] Chunk 7: 'text using a particular algorithm. A simple chunking me'…

[3/3] Generating answer (~1006 context tokens)...
  ✓ Done in 3.73s | ~1,225 tokens

  QUERY PHASE — Page Index RAG
  Question: What are the problems with chunking in vector RAG?

[1/2] Traversing TOC tree...
  Reasoning : The nodes directly relevant to answering the question about problems with chunking in vector RAG are those that discuss the structural flaws and limitations of vector RAG, specifica

In [49]:
YOUR_QUERY = 'How does the solution compare to the problem it was designed to fix?'
yt = your_trad.query(YOUR_QUERY)
yp = your_pi.query(YOUR_QUERY)
print_comparison(yt, yp)


  QUERY PHASE — Traditional RAG
  Question: How does the solution compare to the problem it was designed to fix?

[1/3] Embedding query...

[2/3] Similarity search (top-5)...

  Top-5 similarity scores:
    [0.2880] Chunk 28: 'the final retrieval and generation. To visualize: the u'…
    [0.2855] Chunk 31: 'Cost: Reasoning models consume significantly more token'…
    [0.2770] Chunk 1: "ent retrieval. With that, let's start the video. All ri"…
    [0.2742] Chunk 21: 'at we need to build. But how do we build this tree? You'…
    [0.2738] Chunk 17: 'e between numbers, meaning these numbers heavily rely o'…

[3/3] Generating answer (~1006 context tokens)...
  ✓ Done in 2.78s | ~1,190 tokens

  QUERY PHASE — Page Index RAG
  Question: How does the solution compare to the problem it was designed to fix?

[1/2] Traversing TOC tree...
  Reasoning : The solution, Vectorless RAG, compares to the problem it was designed to fix by addressing the structural flaws and limitations of traditional Ve

In [53]:
!pip install ragas -q
!pip install ragas langchain-openai datasets -q


In [51]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI
from datasets import Dataset

# ── Ground truth answers (you define these manually) ────────
# These are the "correct" answers you expect from the document
eval_data = [
    {
        "question": "What are the problems with chunking in vector RAG?",
        "ground_truth": (
            "The three main problems are: 1) Context Split Problem — "
            "fixed-size chunking cuts concepts mid-sentence losing context. "
            "2) Cross-Reference Problem — chunks miss references to other "
            "pages in the document. 3) Over-reliance on user query "
            "formulation — retrieval quality depends entirely on how well "
            "the user phrases the question."
        ),
    },
    {
        "question": "What is Page Index RAG?",
        "ground_truth": (
            "Page Index RAG is a vectorless, reasoning-based RAG framework "
            "that builds a hierarchical Table of Contents tree from the "
            "document instead of using vector embeddings. It uses LLM "
            "reasoning to traverse the tree and retrieve exact source pages."
        ),
    },
    {
        "question": "What are the trade-offs of Page Index RAG?",
        "ground_truth": (
            "Page Index RAG trades speed and cost for accuracy. "
            "It is slower because LLM reasoning takes longer than vector "
            "lookup, and more expensive because reasoning models consume "
            "more tokens during indexing and traversal."
        ),
    },
]

# ── Build evaluation datasets for both pipelines ────────────

def build_ragas_dataset(pipeline, eval_data):
    rows = []
    for item in eval_data:
        result = pipeline.query(item["question"])
        rows.append({
            "question":        item["question"],
            "answer":          result.answer,
            "contexts":        [result.retrieved_context],
            "ground_truth":    item["ground_truth"],
        })
    return Dataset.from_list(rows)

print("Querying Traditional RAG for evaluation...")
trad_dataset = build_ragas_dataset(your_trad, eval_data)

print("Querying Page Index RAG for evaluation...")
pi_dataset = build_ragas_dataset(your_pi, eval_data)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_10753/18948879.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
/tmp/ipykernel_10753/18948879.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (

Querying Traditional RAG for evaluation...

  QUERY PHASE — Traditional RAG
  Question: What are the problems with chunking in vector RAG?

[1/3] Embedding query...

[2/3] Similarity search (top-5)...

  Top-5 similarity scores:
    [0.6841] Chunk 16: 'n clearly see there is a reference to an entirely diffe'…
    [0.6174] Chunk 13: 'uctural Flaws in Vector RAG Vector RAG works fine; it i'…
    [0.5282] Chunk 14: 'xt. Because you sliced it right down the middle, your c'…
    [0.5249] Chunk 12: 'unk size might be 500 words or a single paragraph, you '…
    [0.5171] Chunk 7: 'text using a particular algorithm. A simple chunking me'…

[3/3] Generating answer (~1006 context tokens)...
  ✓ Done in 3.58s | ~1,274 tokens

  QUERY PHASE — Traditional RAG
  Question: What is Page Index RAG?

[1/3] Embedding query...

[2/3] Similarity search (top-5)...

  Top-5 similarity scores:
    [0.5382] Chunk 0: 'Vectorless RAG (Page Index) – Complete Explanation  Int'…
    [0.5035] Chunk 21: 'at we need to

In [58]:
import time

# ── Custom RAG Evaluator (no RAGAS dependency) ───────────────

eval_llm = LLMClient()

EVAL_QUESTIONS = [
    {
        "question": "What are the problems with chunking in vector RAG?",
        "ground_truth": (
            "The three main problems are: 1) Context Split Problem — "
            "fixed-size chunking cuts concepts mid-sentence losing context. "
            "2) Cross-Reference Problem — chunks miss references to other "
            "pages. 3) Over-reliance on user query formulation — retrieval "
            "quality depends entirely on how well the user phrases the question."
        ),
    },
]


def score_single(question, answer, context, ground_truth, label) -> dict:
    """Ask the LLM to score one answer on all 4 metrics."""

    prompt = f"""You are a RAG evaluation judge. Score the answer on 4 metrics.
Return ONLY a JSON object with scores between 0.0 and 1.0.

QUESTION: {question}
GROUND TRUTH: {ground_truth}
RETRIEVED CONTEXT: {context[:800]}
ANSWER TO EVALUATE: {answer}

Score these metrics:
1. faithfulness      - Is the answer based only on the context? (1=fully grounded, 0=hallucinated)
2. answer_relevancy  - Does the answer actually address the question? (1=fully relevant, 0=off-topic)
3. context_precision - Did the context contain mostly relevant information? (1=precise, 0=noisy)
4. context_recall    - Did the context contain everything needed to answer? (1=complete, 0=missing info)

Output ONLY this JSON, no explanation:
{{"faithfulness": 0.0, "answer_relevancy": 0.0, "context_precision": 0.0, "context_recall": 0.0}}"""

    raw = eval_llm.chat(prompt, system="You are a RAG evaluation judge. Output only valid JSON.")

    # strip markdown fences if present
    if '```' in raw:
        raw = raw.split('```')[1]
        if raw.startswith('json'):
            raw = raw[4:]
    raw = raw.strip()

    try:
        scores = json.loads(raw)
        print(f"    ✓ {label} scored")
        return scores
    except Exception as e:
        print(f"    ⚠ Parse failed for {label}: {e} | raw: {raw[:100]}")
        return {"faithfulness": None, "answer_relevancy": None,
                "context_precision": None, "context_recall": None}


def evaluate_pipeline(pipeline, questions, pipeline_name):
    all_scores = {"faithfulness": [], "answer_relevancy": [],
                  "context_precision": [], "context_recall": []}

    for item in questions:
        print(f'\n  Q: {item["question"][:60]}...')
        result = pipeline.query(item["question"])
        time.sleep(15)  # avoid rate limit between query and scoring

        scores = score_single(
            question=item["question"],
            answer=result.answer,
            context=result.retrieved_context,
            ground_truth=item["ground_truth"],
            label=pipeline_name,
        )
        for metric, val in scores.items():
            if val is not None:
                all_scores[metric].append(val)

        time.sleep(30)  # avoid rate limit between questions

    # average across all questions
    return {
        metric: round(sum(vals) / len(vals), 3) if vals else None
        for metric, vals in all_scores.items()
    }


print('Evaluating Traditional RAG...')
trad_scores = evaluate_pipeline(your_trad, EVAL_QUESTIONS, 'Traditional RAG')

print('\nWaiting 20s before Page Index evaluation...')
time.sleep(20)

print('\nEvaluating Page Index RAG...')
pi_scores = evaluate_pipeline(your_pi, EVAL_QUESTIONS, 'Page Index RAG')

# ── Final scorecard ──────────────────────────────────────────
print('\n' + '═'*65)
print('  EVALUATION SCORECARD (0 = worst, 1 = best)')
print('═'*65)
print(f'  {"Metric":<22} {"Traditional RAG":<20} {"Page Index RAG":<16} {"Winner"}')
print('  ' + '-'*60)

for metric in ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']:
    t = trad_scores.get(metric)
    p = pi_scores.get(metric)
    t_str = f'{t:.3f}' if t is not None else 'failed'
    p_str = f'{p:.3f}' if p is not None else 'failed'
    if t is not None and p is not None:
        winner = 'Page Index' if p > t else 'Traditional' if t > p else 'Tie'
    else:
        winner = 'N/A'
    print(f'  {metric:<22} {t_str:<20} {p_str:<16} {winner}')

print('═'*65)

Evaluating Traditional RAG...

  Q: What are the problems with chunking in vector RAG?...

  QUERY PHASE — Traditional RAG
  Question: What are the problems with chunking in vector RAG?

[1/3] Embedding query...

[2/3] Similarity search (top-5)...

  Top-5 similarity scores:
    [0.6841] Chunk 16: 'n clearly see there is a reference to an entirely diffe'…
    [0.6174] Chunk 13: 'uctural Flaws in Vector RAG Vector RAG works fine; it i'…
    [0.5282] Chunk 14: 'xt. Because you sliced it right down the middle, your c'…
    [0.5249] Chunk 12: 'unk size might be 500 words or a single paragraph, you '…
    [0.5171] Chunk 7: 'text using a particular algorithm. A simple chunking me'…

[3/3] Generating answer (~1006 context tokens)...
  ✓ Done in 13.52s | ~1,232 tokens
    ✓ Traditional RAG scored

Waiting 20s before Page Index evaluation...

Evaluating Page Index RAG...

  Q: What are the problems with chunking in vector RAG?...

  QUERY PHASE — Page Index RAG
  Question: What are the problems

**Traditional RAG retrieved 5 chunks** with strong similarity scores (0.51–0.68), meaning it found genuinely relevant content this time — but still scored only 0.6 on faithfulness, indicating the LLM mixed retrieved facts with its own knowledge to fill gaps between the fragmented chunks.

**Page Index **precisely identified 3 nodes (node_006 and its two children) through structural reasoning, correctly navigating to the "Structural Flaws" section of the tree directly — resulting in 0.9 faithfulness, meaning 90% of the answer came purely from the source document with minimal hallucination.

**Page Index** won on all 4 metrics, with the biggest gap on faithfulness (0.6 vs 0.9) and answer relevancy (0.8 vs 1.0) — confirming that for well-structured documents with clear section headings, reasoning-based retrieval produces more grounded and complete answers than keyword similarity search.

**The context recall scores** (0.6 and 0.7) being the lowest for both pipelines indicates neither retrieved everything needed for a fully complete answer — adding more questions to the evaluation set would give a more reliable picture of true recall performance.

## Key takeaways

In [48]:
print("""
╔══════════════════════════════════════════════════════════════╗
║            RAG COMPARISON — WHAT YOU LEARNED                ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  TRADITIONAL RAG                                             ║
║  ✓ Fast retrieval  (vector lookup = simple dot product)      ║
║  ✓ Low cost per query (embed once, search cheaply)           ║
║  ✓ Works on any unstructured text                            ║
║  ✗ Fixed chunks cut context mid-concept or mid-sentence      ║
║  ✗ Retrieval quality = query keyword quality                 ║
║  ✗ Cross-references across sections are usually missed       ║
║                                                              ║
║  PAGE INDEX RAG                                              ║
║  ✓ Reasoning-based — vague queries handled better            ║
║  ✓ Structure-aware — respects natural section boundaries     ║
║  ✓ Cross-references: tree nodes can point to any page        ║
║  ✗ Expensive indexing (multiple LLM calls per page batch)    ║
║  ✗ Slower query (reasoning takes longer than dot product)    ║
║  ✗ Unstructured docs produce poor trees                      ║
║  ✗ Bad index node = all future queries on that section fail  ║
║                                                              ║
║  WHEN TO USE WHICH                                           ║
║  Traditional : large corpora, cost-sensitive, varied docs    ║
║  Page Index  : books, legal docs, manuals, structured PDFs   ║
╚══════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════╗
║            RAG COMPARISON — WHAT YOU LEARNED                ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  TRADITIONAL RAG                                             ║
║  ✓ Fast retrieval  (vector lookup = simple dot product)      ║
║  ✓ Low cost per query (embed once, search cheaply)           ║
║  ✓ Works on any unstructured text                            ║
║  ✗ Fixed chunks cut context mid-concept or mid-sentence      ║
║  ✗ Retrieval quality = query keyword quality                 ║
║  ✗ Cross-references across sections are usually missed       ║
║                                                              ║
║  PAGE INDEX RAG                                              ║
║  ✓ Reasoning-based — vague queries handled better            ║
║  ✓ Structure-aware — respects natural section boundaries     ║
║  ✓ Cross-references: tr